In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)

BASE_DIR = Path("/home/hp/projects/lending_club_fin")
DATA_PATH = BASE_DIR / "data/raw/data_80.csv"

In [ ]:
import sys
sys.path.insert(0, str(BASE_DIR / "scripts"))

from ingestion import load_raw_data
from cleaning import clean_pipeline
from transformation import transform_pipeline

df_raw = load_raw_data(str(DATA_PATH))
print(f"Shape: {df_raw.shape}")
df_raw.head()

In [ ]:
def basic_info(df):
    print("=== SHAPE ===")
    print(df.shape)
    print("\n=== DTYPES ===")
    print(df.dtypes.value_counts())
    print("\n=== MISSING VALUES (top 20) ===")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({"missing": missing, "pct": missing_pct})
    print(missing_df[missing_df["missing"] > 0].sort_values("pct", ascending=False).head(20))

basic_info(df_raw)

In [ ]:
def plot_missing(df, top_n=30):
    missing_pct = (df.isnull().sum() / len(df) * 100)
    missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False).head(top_n)
    
    plt.figure(figsize=(14, 6))
    sns.barplot(x=missing_pct.values, y=missing_pct.index, palette="Reds_r")
    plt.title(f"Top {top_n} Columns by Missing Value %", fontsize=14)
    plt.xlabel("Missing %")
    plt.tight_layout()
    plt.show()

plot_missing(df_raw)

In [ ]:
def plot_loan_status(df):
    status_counts = df["loan_status"].value_counts()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar chart
    sns.barplot(x=status_counts.values, y=status_counts.index, palette="viridis", ax=axes[0])
    axes[0].set_title("Loan Status Counts")
    axes[0].set_xlabel("Count")
    
    # Pie chart
    axes[1].pie(status_counts.values, labels=status_counts.index, autopct="%1.1f%%", startangle=140)
    axes[1].set_title("Loan Status Distribution")
    
    plt.tight_layout()
    plt.show()
    
    print("\nDefault-related statuses:")
    default_statuses = ["Charged Off", "Default", "Does not meet the credit policy. Status:Charged Off"]
    default_pct = df["loan_status"].isin(default_statuses).mean() * 100
    print(f"  Default rate: {default_pct:.2f}%")

plot_loan_status(df_raw)

In [ ]:
def plot_loan_amount(df):
    df["loan_amnt"] = pd.to_numeric(df["loan_amnt"], errors="coerce")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    sns.histplot(df["loan_amnt"].dropna(), bins=50, kde=True, ax=axes[0], color="steelblue")
    axes[0].set_title("Loan Amount Distribution")
    axes[0].set_xlabel("Loan Amount ($)")
    
    sns.boxplot(x=df["loan_amnt"].dropna(), ax=axes[1], color="steelblue")
    axes[1].set_title("Loan Amount Boxplot")
    
    plt.tight_layout()
    plt.show()
    
    print(df["loan_amnt"].describe())

plot_loan_amount(df_raw)

In [ ]:
def plot_interest_by_grade(df):
    df["int_rate"] = pd.to_numeric(
        df["int_rate"].astype(str).str.replace("%", "").str.strip(), errors="coerce"
    )
    
    grade_order = sorted(df["grade"].dropna().unique())
    
    plt.figure(figsize=(12, 5))
    sns.boxplot(data=df, x="grade", y="int_rate", order=grade_order, palette="coolwarm")
    plt.title("Interest Rate Distribution by Loan Grade")
    plt.xlabel("Grade")
    plt.ylabel("Interest Rate (%)")
    plt.tight_layout()
    plt.show()

plot_interest_by_grade(df_raw)

In [ ]:
def plot_default_by_grade(df):
    default_statuses = ["Charged Off", "Default", "Does not meet the credit policy. Status:Charged Off"]
    df["is_default"] = df["loan_status"].isin(default_statuses).astype(int)
    
    default_by_grade = df.groupby("grade")["is_default"].mean() * 100
    default_by_grade = default_by_grade.sort_index()
    
    plt.figure(figsize=(10, 5))
    sns.barplot(x=default_by_grade.index, y=default_by_grade.values, palette="Reds")
    plt.title("Default Rate by Loan Grade (%)")
    plt.xlabel("Grade")
    plt.ylabel("Default Rate (%)")
    plt.tight_layout()
    plt.show()

plot_default_by_grade(df_raw)

In [ ]:
def plot_loan_volume_over_time(df):
    df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")
    df["issue_year"] = df["issue_d"].dt.year
    df["issue_month"] = df["issue_d"].dt.month
    
    yearly = df.groupby("issue_year")["loan_amnt"].agg(["count", "sum"]).reset_index()
    yearly.columns = ["year", "count", "total_amount"]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    sns.lineplot(data=yearly, x="year", y="count", marker="o", ax=axes[0], color="steelblue")
    axes[0].set_title("Number of Loans Issued Per Year")
    axes[0].set_xlabel("Year")
    axes[0].set_ylabel("Count")
    
    sns.lineplot(data=yearly, x="year", y="total_amount", marker="o", ax=axes[1], color="green")
    axes[1].set_title("Total Loan Amount Issued Per Year ($)")
    axes[1].set_xlabel("Year")
    axes[1].set_ylabel("Total Amount")
    
    plt.tight_layout()
    plt.show()

plot_loan_volume_over_time(df_raw)

In [ ]:
def plot_fico_by_default(df):
    default_statuses = ["Charged Off", "Default", "Does not meet the credit policy. Status:Charged Off"]
    df["is_default"] = df["loan_status"].isin(default_statuses).astype(int)
    df["fico_range_low"] = pd.to_numeric(df["fico_range_low"], errors="coerce")
    df["fico_mid"] = (pd.to_numeric(df["fico_range_low"], errors="coerce") + 
                      pd.to_numeric(df["fico_range_high"], errors="coerce")) / 2
    
    plt.figure(figsize=(12, 5))
    sns.kdeplot(data=df[df["is_default"]==0], x="fico_mid", label="Non-Default", fill=True, alpha=0.5)
    sns.kdeplot(data=df[df["is_default"]==1], x="fico_mid", label="Default", fill=True, alpha=0.5, color="red")
    plt.title("FICO Score Distribution: Default vs Non-Default")
    plt.xlabel("FICO Midpoint Score")
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_fico_by_default(df_raw)

In [ ]:
def plot_dti_by_default(df):
    default_statuses = ["Charged Off", "Default", "Does not meet the credit policy. Status:Charged Off"]
    df["is_default"] = df["loan_status"].isin(default_statuses).astype(int)
    df["dti"] = pd.to_numeric(df["dti"], errors="coerce")
    df_filtered = df[df["dti"] < 60]
    
    plt.figure(figsize=(12, 5))
    sns.kdeplot(data=df_filtered[df_filtered["is_default"]==0], x="dti", label="Non-Default", fill=True, alpha=0.5)
    sns.kdeplot(data=df_filtered[df_filtered["is_default"]==1], x="dti", label="Default", fill=True, alpha=0.5, color="red")
    plt.title("DTI Distribution: Default vs Non-Default")
    plt.xlabel("Debt-to-Income Ratio")
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_dti_by_default(df_raw)

In [ ]:
def plot_loan_purpose(df):
    purpose_counts = df["purpose"].value_counts().head(10)
    
    plt.figure(figsize=(12, 5))
    sns.barplot(x=purpose_counts.values, y=purpose_counts.index, palette="Blues_r")
    plt.title("Top 10 Loan Purposes")
    plt.xlabel("Count")
    plt.tight_layout()
    plt.show()

plot_loan_purpose(df_raw)

In [ ]:
def plot_correlation(df):
    df_clean = clean_pipeline(df.copy())
    
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
    corr = df_clean[numeric_cols].corr()
    
    plt.figure(figsize=(14, 10))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", 
                linewidths=0.5, annot_kws={"size": 8})
    plt.title("Correlation Heatmap of Numeric Features")
    plt.tight_layout()
    plt.show()

plot_correlation(df_raw)

In [ ]:
def plot_homeownership_default(df):
    default_statuses = ["Charged Off", "Default", "Does not meet the credit policy. Status:Charged Off"]
    df["is_default"] = df["loan_status"].isin(default_statuses).astype(int)
    
    home_default = df.groupby("home_ownership")["is_default"].mean() * 100
    home_default = home_default.sort_values(ascending=False)
    
    plt.figure(figsize=(10, 5))
    sns.barplot(x=home_default.index, y=home_default.values, palette="Set2")
    plt.title("Default Rate by Home Ownership (%)")
    plt.xlabel("Home Ownership")
    plt.ylabel("Default Rate (%)")
    plt.tight_layout()
    plt.show()

plot_homeownership_default(df_raw)

In [ ]:
def summary_insights(df):
    default_statuses = ["Charged Off", "Default", "Does not meet the credit policy. Status:Charged Off"]
    df["is_default"] = df["loan_status"].isin(default_statuses).astype(int)
    df["loan_amnt"] = pd.to_numeric(df["loan_amnt"], errors="coerce")
    df["int_rate"] = pd.to_numeric(df["int_rate"].astype(str).str.replace("%","").str.strip(), errors="coerce")
    df["annual_inc"] = pd.to_numeric(df["annual_inc"], errors="coerce")

    print("=" * 50)
    print("KEY INSIGHTS")
    print("=" * 50)
    print(f"Total loans:          {len(df):,}")
    print(f"Overall default rate: {df['is_default'].mean()*100:.2f}%")
    print(f"Avg loan amount:      ${df['loan_amnt'].mean():,.2f}")
    print(f"Avg interest rate:    {df['int_rate'].mean():.2f}%")
    print(f"Avg annual income:    ${df['annual_inc'].mean():,.2f}")
    print(f"Date range:           {df['issue_d'].min()} → {df['issue_d'].max()}")
    print(f"Most common purpose:  {df['purpose'].mode()[0]}")
    print(f"Most common grade:    {df['grade'].mode()[0]}")

summary_insights(df_raw)